In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import os
from tqdm import tqdm
PATH_SOURCE = Path("/home/vlitoux/Documents/play-diverse/data")
PATH_INPUT = Path("./input")
PATH_AIS_250_VESSELS_INPUT = PATH_INPUT / "ais_250_vessels"
PATH_AIS_5_VESSELS_INPUT = PATH_INPUT / "ais_5_vessels"
%load_ext autoreload
%autoreload 2

In [2]:
def get_memory_usages(df_input: pd.DataFrame, ascending=False, sort_col='size_KB'):
	"""
	Get a overview of a dataframe (category, number unics, and size in KB).
	Sort according to 'sort_col'
	"""
	m_usage = df_input.memory_usage(deep=True) / 1024
	# Concatenate with unique
	uniques_df = df_input.nunique()
	return (
		pd.concat([m_usage, uniques_df, df_input.dtypes], axis=1)
		.rename(columns={0: 'size_KB', 1: 'distinct_number', 2: 'type'})
		.sort_values(sort_col, ascending=ascending)
	)

### Get list of MMSI

In [ ]:
gdf_input = gpd.read_parquet(PATH_SOURCE / "ais-2024-07-02.parquet",columns=["mmsi"])
list_sample_mmsi : pd.DataFrame = gdf_input.groupby("mmsi").size().sort_values(ascending=False)
list_sample_mmsi.to_csv(PATH_INPUT / "list_mmsi.csv")
list_sample_mmsi


### filter on all files to get only a sample

In [3]:
list_sample_mmsi = pd.read_csv(PATH_INPUT / "list_mmsi.csv")[:250]
list_sample_mmsi

,mmsi,0
0,440800014,1405
1,338199855,1392
2,367616260,1388
3,367458840,1376
4,368311450,1374
...,...,...
245,368031660,1302
246,367189940,1302
247,367465980,1301
248,367313850,1301


##### First with small number mmsi (250)

In [8]:

list_mmsi_col = list_sample_mmsi["mmsi"]

def read_and_filter(path_input: str, file_name_input: str, path_output: str):
    gpd.read_parquet(path_input /file_name_input ,columns=["mmsi", "base_date_time", "geometry","heading","sog"]).query("mmsi in @list_mmsi_col").to_parquet(path_output / f"filtered-{file_name_input}")



In [ ]:
for item in tqdm(os.listdir(PATH_SOURCE)):
    read_and_filter(path_input= PATH_SOURCE, file_name_input=item, path_output=PATH_AIS_250_VESSELS_INPUT)

##### Seconds with a couple (5)

In [11]:

list_mmsi_col_5_vessels = list_sample_mmsi["mmsi"][:5]

def read_and_filter(path_input: str, file_name_input: str, path_output: str):
    gpd.read_parquet(path_input /file_name_input ,columns=["mmsi", "base_date_time", "geometry","heading","sog"]).query("mmsi in @list_mmsi_col_5_vessels").to_parquet(path_output / f"filtered-{file_name_input}")


for item in tqdm(os.listdir(PATH_AIS_250_VESSELS_INPUT)):
    read_and_filter(path_input= PATH_AIS_250_VESSELS_INPUT, file_name_input=item, path_output=PATH_AIS_5_VESSELS_INPUT)

100%|██████████| 366/366 [01:22<00:00,  4.46it/s]


### Aggregate all files

In [28]:
gdf_all = gpd.read_parquet(PATH_AIS_5_VESSELS_INPUT).rename(columns={"base_date_time": "timestamp"})
get_memory_usages(gdf_all)

,size_KB,distinct_number,type
timestamp,8768.868164,1084248.0,timestamp[s][pyarrow]
Index,8633.804688,NaN,NaN
geometry,8633.804688,174017.0,geometry
mmsi,4451.965820,5.0,int32[pyarrow]
heading,4451.965820,360.0,int32[pyarrow]
sog,4451.965820,343.0,float[pyarrow]


In [7]:
gdf_all.groupby("mmsi").size()

mmsi
338199855    456733
367458840    411201
367616260    113021
368311450     82150
440800014     42022
dtype: int64

In [34]:
import movingpandas as mpd
import colorcet as cc
from holoviews import Overlay
import hvplot

Category20_20 = (
	'#1f77b4',
	'#aec7e8',
	'#ff7f0e',
	'#ffbb78',
	'#2ca02c',
	'#98df8a',
	'#d62728',
	'#ff9896',
	'#9467bd',
	'#c5b0d5',
	'#8c564b',
	'#c49c94',
	'#e377c2',
	'#f7b6d2',
	'#7f7f7f',
	'#c7c7c7',
	'#bcbd22',
	'#dbdb8d',
	'#17becf',
	'#9edae5',
)
MPD_PALETTE = list(Category20_20) + cc.palette['glasbey']
default_width = 800
default_height = 600
traj= mpd.TrajectoryCollection(
 		gdf_all.sort_values("timestamp"), t='timestamp',traj_id_col="mmsi"
	)

traj


TrajectoryCollection with 5 trajectories

In [45]:
traj_ex : mpd.Trajectory = traj.get_trajectory(440800014)
traj_ex.add_distance()

Trajectory 440800014 (2024-06-10 21:33:43 to 2024-08-31 08:00:14) | Size: 42020 | Length: 17791574.2m
Bounds: (-159.98998, 11.80235, 145.16558, 23.46754)
LINESTRING (144.02895 13.86802, 144.04772 13.85458, 144.10983 13.80874, 144.11867 13.80202, 144.1224

In [46]:
traj_ex.df

,mmsi,geometry,heading,sog,distance
timestamp,,,,,
2024-06-10 21:33:43,440800014,POINT (144.02895 13.86802),<NA>,14.0,0.000000
2024-06-10 21:39:26,440800014,POINT (144.04772 13.85458),<NA>,14.4,2515.545643
2024-06-10 21:58:44,440800014,POINT (144.10983 13.80874),<NA>,13.9,8414.906339
2024-06-10 22:01:32,440800014,POINT (144.11867 13.80202),<NA>,13.8,1210.932776
2024-06-10 22:02:44,440800014,POINT (144.12242 13.79913),<NA>,13.9,516.376454
...,...,...,...,...,...
2024-08-31 07:52:33,440800014,POINT (144.54951 14.00292),<NA>,10.2,416.109342
2024-08-31 07:55:14,440800014,POINT (144.55378 14.00912),<NA>,9.8,826.642543
2024-08-31 07:56:33,440800014,POINT (144.55579 14.01221),<NA>,9.7,405.003760


In [50]:
traj_ex

Trajectory 440800014 (2024-06-10 21:33:43 to 2024-08-31 08:00:14) | Size: 42020 | Length: 17791574.2m
Bounds: (-159.98998, 11.80235, 145.16558, 23.46754)
LINESTRING (144.02895 13.86802, 144.04772 13.85458, 144.10983 13.80874, 144.11867 13.80202, 144.1224

In [55]:
new_traj_coll = mpd.DouglasPeuckerGeneralizer(traj).generalize(0.8)
new_traj_coll

TrajectoryCollection with 5 trajectories

In [56]:
new_traj_coll.hvplot(
		tiles='CartoLight',
		legend=True,
        hover_cols=["mmsi"],
		width=default_width,
		height=default_height,

	)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Path.I     :Path   [Longitude,Latitude]   (mmsi)
   .Path.II    :Path   [Longitude,Latitude]   (mmsi)
   .Path.III   :Path   [Longitude,Latitude]   (mmsi)
   .Path.IV    :Path   [Longitude,Latitude]   (mmsi)
   .Path.V     :Path   [Longitude,Latitude]   (mmsi)
   .Points.I   :Points   [Longitude,Latitude]   (triangle_angle,mmsi)
   .Points.II  :Points   [Longitude,Latitude]   (triangle_angle,mmsi)
   .Points.III :Points   [Longitude,Latitude]   (triangle_angle,mmsi)
   .Points.IV  :Points   [Longitude,Latitude]   (triangle_angle,mmsi)
   .Points.V   :Points   [Longitude,Latitude]   (triangle_angle,mmsi)

In [ ]:
gdf_

# Ship navi sim study

In [57]:
gdf_all

,mmsi,timestamp,geometry,heading,sog
711,338199855,2024-01-01 00:00:00,POINT (-158.11989 21.32865),256,0.0
2064,367458840,2024-01-01 00:00:02,POINT (-80.14435 25.76414),354,0.4
9651,367458840,2024-01-01 00:02:08,POINT (-80.14435 25.76412),353,0.0
9667,367458840,2024-01-01 00:01:05,POINT (-80.14437 25.76412),176,0.0
12249,338199855,2024-01-01 00:01:02,POINT (-158.11989 21.32865),252,0.0
...,...,...,...,...,...
7558341,367458840,2024-12-31 23:30:08,POINT (-80.1448 25.76444),112,0.1
7558376,367458840,2024-12-31 23:32:18,POINT (-80.14479 25.76443),87,0.0
7573527,367458840,2024-12-31 23:49:42,POINT (-80.1448 25.76445),89,0.1
7573573,367458840,2024-12-31 23:52:57,POINT (-80.14479 25.76442),89,0.0


In [59]:
gdf_all["sog"].describe()

count    1105125.0
mean      1.148688
std        2.81075
min            0.0
25%            0.0
50%            0.0
75%            0.0
max      36.799999
Name: sog, dtype: double[pyarrow]

In [62]:
import gymnasium as gym
from ship_env import ShipEnvironment

# Prepare trajectories, times and overlap_idx (see data preparation section)
import pyrallis
from il_bc import TrainConfig, load_ships_and_play

config = TrainConfig(
    mode="train",
    env="Maritime-Expert-v1",
    batch_size=256,
    use_det=True,
    wandb=False
)

load_ships_and_play(config)

usage: ipykernel_launcher.py [-h] [--config_path str] [--mode str]
                             [--project str] [--group str] [--name str]
                             [--env str] [--use_combMLP str] [--use_det str]
                             [--max_timesteps str] [--batch_size str]
                             [--use_her str] [--seed str] [--device str]
                             [--wandb str] [--ckpoint_folder str]
                             [--ckpoint_path str]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1000/jupyter/runtime/kernel-v3645ff351326f9f6455f3dd7c87f97a2aabd735ed.json


SystemExit: 2

/home/vlitoux/miniconda3/envs/maritime-rl/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
